# 06 Supervised models w/ AI features
> By Tahreem Karim

Does AI-policy add signal, beyond infrastructure & geography? **Model task:** binary classification per county, *does it host (at least) one IM3 atlas facility*? Since the experiment compares feature sets not models, *four model families are each trained on three nested feature sets* (defined in `pipeline_config.py`):

| Set | Features | Tests |
|---|---|---|
| Infrastructure | 11 CERF-DC county features | How do PNNL's own inputs perform when learned from observed sites (see below) |
| + Geography | + Census division (9 one-hot columns) | How much a county's region explains, w/o policy content |
| + AI policy | + 9 state AI-policy features (notebook 05) | Whether policy adds anything beyond geography |

Relation to PNNL's study
- PNNL’s CERF-DC tool scores counties using a **fixed, hand built formula** over infrastructure variables: land value, substation distance, power price, tax rates, cooling, etc.. Basically it’s **not learned** from data, but our model is.

- The “Infrastructure” feature, shown in the first row of the table above, uses the exact same 11 variables PNNL’s tool does, but *instead of running a formula we train a model on them which learns from where real data center locations* (Atlas records). We are seeing whether PNNL’s inputs can predict real world sites, if you let a model learn the relationship statistically instead of using PNNL’s fixed scoring rule. That's the baseline, the next two rows ask how much more insightful adding region & policy is (data PNNL doesn’t use). The analysis will be in the next notebook.

Validation 
- Nested, state grouped cross-validation: 5 outer folds × 3 seeds (15 test folds), w/ a
3-fold state grouped grid search inside each training split. Counties from one state are never in both
train and test, so a state level feature cannot score by memorizing a state.
- **Failure testing** in next notebook.

Metrics
- **PR-AUC** (primary b/c only 7.9% of counties have a data center)
- **ROC-AUC**
- **Brier** score on probabilities calibrated by crossfit isotonic regression (class weighting inflates raw probabilities), see 'Calibration (cross-fitted isotonic)' cell in middle of ntbk

#### Outputs `processed/stage4_supervised/`: `fold_scores.csv`, `oof_predictions.csv`, `best_params.csv`,
`fold_models.pkl` (fitted XGBoost fold models, reused for SHAP in notebook 07).

In [ ]:
import pickle
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

from pipeline_config import FEATURE_SETS, GROUP, STAGE4, TARGET, load_model_table, make_models, outer_splits

warnings.filterwarnings("ignore", category=UserWarning)
RERUN = False            # True refits everything (~ 10 minutes on 2 cores), False reuses saved results

d = load_model_table()
y, groups = d[TARGET].values, d[GROUP].values
SPW = (y == 0).sum() / y.sum()          # XGBoost class weight: negatives per positive

print(f"{len(d):,} counties, {y.sum()} positive ({y.mean():.1%}), {len(set(groups))} states; scale_pos_weight {SPW:.2f}")

3,099 counties, 245 positive (7.9%), 48 states; scale_pos_weight 11.65


## Model families & grids

| Family | Mechanism | Why it's here |
|---|---|---|
| Logistic regression | linear, probabilistic | Signed coefficients a policy reader can check; class weights handle imbalance |
| k-nearest neighbors | instance based | Asks whether data center counties sit close together in infrastructure space |
| Balanced random forest | bagged trees on class balanced bootstraps | Threshold effects & interactions, imbalance handled w/o synthetic data |
| XGBoost | boosted trees | Usually strongest on tabular data; exact TreeSHAP for ntbk 07 |

Linear and kNN models see standardized features (after the electricity cap and log of skewed columns).
Trees see the same capped, logged values unscaled. The pipelines and grids are defined once in
`pipeline_config.make_models` so notebook 07 reuses them.

In [ ]:
# inner splits fx to tune hyperparameters
def inner_cv(y_tr, g_tr, seed):
    return list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed).split(np.zeros(len(y_tr)), y_tr, g_tr))

## Nested, state-grouped cross validation
Nests two rounds of state grouped cross val. Inner round tunes each model’s hyperparameters using only training states, inside outer round testing tuned model on states it hasn't seen. Held out test fold stays untouched by fitting/tuning. 
>**"Test folds"** - to check if a model generalizes, you can’t test on the same counties it trained on (data leakage --> overfitting). Counties get split into groups (or "folds”), model trains on most of them & gets tested on held out ones. Our splits are grouped by state, meaning a whole state’s counties are held out together (not split btw. training/testing). This is repeated 5x w/ 3 diff. random seeds.

`for` loops:
1. Five state groupings × three random seeds. 
2. Three feature sets (infrastructure only / + region / + policy). Repeats test w/ progressively more information
3. Four model types

In [ ]:
oof_file, params_file, models_file = STAGE4 / "oof_raw.csv", STAGE4 / "best_params.csv", STAGE4 / "fold_models.pkl"

if RERUN or not oof_file.exists():                                  # refit or reuse saved results
    rows, params, fold_models, t0 = [], [], {}, time.time()
    
    for seed, k, tr, te in outer_splits(y, groups):
        icv = inner_cv(y[tr], groups[tr], seed)
        
        for set_name, cols in FEATURE_SETS.items():
            X = d[cols]
            
            for fam, (pipe, grid) in make_models(cols, SPW).items():
                gs = GridSearchCV(pipe, grid, scoring="average_precision", cv=icv, n_jobs=2, refit=True)
                gs.fit(X.iloc[tr], y[tr])
                score = gs.predict_proba(X.iloc[te])[:, 1]
                rows.append(pd.DataFrame({"seed": seed, "fold": k, "set": set_name, "model": fam,
                                          "row": te, "y": y[te], "raw": score}))
                params.append({"seed": seed, "fold": k, "set": set_name, "model": fam,
                               "inner_pr_auc": gs.best_score_, **{p.split("__")[1]: v for p, v in gs.best_params_.items()}})
                
                if fam == "XGBoost":
                    fold_models[(seed, k, set_name)] = gs.best_estimator_
       
        print(f"seed {seed} fold {k} done, {time.time() - t0:.0f}s")
    pd.concat(rows).to_csv(oof_file, index=False)
    pd.DataFrame(params).to_csv(params_file, index=False)
    pickle.dump(fold_models, open(models_file, "wb"))

# "out of fold"
oof = pd.read_csv(oof_file)
params = pd.read_csv(params_file)
print(f"{len(oof):,} out of fold predictions ({oof.groupby(['seed', 'set', 'model']).ngroups} seed × set × model runs)")

seed 0 fold 0 done, 37s


seed 0 fold 1 done, 72s


seed 0 fold 2 done, 105s


seed 0 fold 3 done, 140s


seed 0 fold 4 done, 175s


...

seed 2 fold 1 done, 421s


seed 2 fold 2 done, 456s


seed 2 fold 3 done, 492s


seed 2 fold 4 done, 528s


111,564 out-of-fold predictions (36 seed × set × model runs)


## Calibration (cross-fitted isotonic)

*The models were designed to treat the rare class as more important* because of the imbalance stemming from 245 data centers existing in 3,099 counties (**7.9%**), so raw outputs skew confident. For each test fold, an isotonic map is fitted on the other folds 'out of fold scores' from the same seed,
then applied to this fold. Calibrator never sees county it calibrates.

In [ ]:
def cross_fit_isotonic(g):
    out = pd.Series(index=g.index, dtype=float)
    for k in g.fold.unique():
        rest, this = g[g.fold != k], g.fold == k
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(rest.raw, rest.y)
        out[this] = iso.predict(g.loc[this, "raw"])
    return out


oof["prob"] = oof.groupby(["seed", "set", "model"], group_keys=False).apply(cross_fit_isotonic, include_groups=False)
oof["county_fips"] = d.county_fips.values[oof.row]
oof["state_abbr"] = d.state_abbr.values[oof.row]
oof.to_csv(STAGE4 / "oof_predictions.csv", index=False)

## Scores per test fold & reference baselines (no learning)
Scores every model on every test fold & builds two 'dumb' baselines (guessing overall positive rate w/ no model) to compare real models against.

In [ ]:
def fold_metrics(g):
    return pd.Series({"pr_auc": average_precision_score(g.y, g.raw), "roc_auc": roc_auc_score(g.y, g.raw),
                      "brier": brier_score_loss(g.y, g.prob), "brier_raw": brier_score_loss(g.y, g.raw.clip(0, 1)),
                      "positives": int(g.y.sum()), "counties": len(g)})


scores = oof.groupby(["set", "model", "seed", "fold"]).apply(fold_metrics, include_groups=False).reset_index()

base = []
for seed, k, tr, te in outer_splits(y, groups):
    yt = y[te]
    base.append({"set": "Reference", "model": "No skill (positive rate)", "seed": seed, "fold": k,
                 "pr_auc": yt.mean(), "roc_auc": 0.5, "brier": brier_score_loss(yt, np.full(len(yt), y[tr].mean()))})

    s = d.composite_siting_suitability_mean.values[te]
    base.append({"set": "Reference", "model": "CERF composite suitability ranking", "seed": seed, "fold": k,
                 "pr_auc": average_precision_score(yt, s), "roc_auc": roc_auc_score(yt, s), "brier": np.nan})

scores = pd.concat([scores, pd.DataFrame(base)], ignore_index=True)
scores.to_csv(STAGE4 / "fold_scores.csv", index=False)
summary = (scores.groupby(["set", "model"])[["pr_auc", "roc_auc", "brier"]].agg(["mean", "std"]).round(3))
summary

pr_auc        roc_auc  \
                                                    mean    std    mean   
set            model                                                      
+ AI policy    Balanced random forest              0.370  0.065   0.801   
               Logistic regression                 0.336  0.068   0.774   
               XGBoost                             0.381  0.062   0.804   
               k-nearest neighbors                 0.265  0.074   0.748   
+ Geography    Balanced random forest              0.359  0.070   0.807   
               Logistic regression                 0.359  0.069   0.799   
               XGBoost                             0.372  0.059   0.803   
               k-nearest neighbors                 0.274  0.053   0.735   
Infrastructure Balanced random forest              0.349  0.064   0.788   
               Logistic regression                 0.353  0.061   0.791   
               XGBoost                             0.362  0.057   0.787   
               k-nearest neighbors                 0.279  0.067   0.733   
Reference      CERF composite suitability ranking  0.187  0.055   0.677   
               No skill (positive rate)            0.079  0.004   0.500   

                                                          brier         
                                                     std   mean    std  
set            model                                                    
+ AI policy    Balanced random forest              0.042  0.064  0.009  
               Logistic regression                 0.060  0.066  0.004  
               XGBoost                             0.040  0.062  0.005  
               k-nearest neighbors                 0.063  0.072  0.008  
+ Geography    Balanced random forest              0.040  0.065  0.008  
               Logistic regression                 0.053  0.065  0.003  
               XGBoost                             0.038  0.063  0.005  
               k-nearest neighbors                 0.052  0.071  0.009  
Infrastructure Balanced random forest              0.043  0.065  0.008  
               Logistic regression                 0.045  0.064  0.004  
               XGBoost                             0.041  0.063  0.005  
               k-nearest neighbors                 0.056  0.069  0.005  
Reference      CERF composite suitability ranking  0.049    NaN    NaN  
               No skill (positive rate)            0.000  0.073  0.004

## Table A2 · PR-AUC by feature set & paired test for policy

Every family & feature set is scored on same 15 folds, so the policy effect is measured as a
**paired difference** per fold: (+ AI policy) − (+ Geography).

In [ ]:
order = list(FEATURE_SETS)
fam_order = ["Logistic regression", "k-nearest neighbors", "Balanced random forest", "XGBoost"]
pr = scores[scores.set.isin(order)].pivot_table(index=["model", "seed", "fold"], columns="set", values="pr_auc")
fmt = lambda s: f"{s.mean():.3f} ± {s.std():.3f}"

table_a2 = pd.DataFrame({c: pr[c].groupby("model").apply(fmt) for c in order}).loc[fam_order]
deltas = pd.DataFrame({
    "geo_minus_infra": pr["+ Geography"] - pr["Infrastructure"],
    "policy_minus_geo": pr["+ AI policy"] - pr["+ Geography"]})

paired = deltas.groupby("model").agg(
    geo_gain=("geo_minus_infra", "mean"), geo_gain_sd=("geo_minus_infra", "std"),
    policy_gain=("policy_minus_geo", "mean"), policy_gain_sd=("policy_minus_geo", "std"),
    policy_wins=("policy_minus_geo", lambda s: f"{(s > 0).sum()} of {len(s)}")).loc[fam_order].round(3)

paired.to_csv(STAGE4 / "paired_policy_test.csv")
table_a2.join(paired)

,Infrastructure,+ Geography,+ AI policy,geo_gain,geo_gain_sd,policy_gain,policy_gain_sd,policy_wins
model,,,,,,,,
Logistic regression,0.353 ± 0.061,0.359 ± 0.069,0.336 ± 0.068,0.006,0.027,-0.023,0.043,5 of 15
k-nearest neighbors,0.279 ± 0.067,0.274 ± 0.053,0.265 ± 0.074,-0.005,0.058,-0.009,0.044,4 of 15
Balanced random forest,0.349 ± 0.064,0.359 ± 0.070,0.370 ± 0.065,0.010,0.024,0.011,0.028,8 of 15
XGBoost,0.362 ± 0.057,0.372 ± 0.059,0.381 ± 0.062,0.011,0.025,0.009,0.030,9 of 15


## Table A1 · Best model per family (+ AI policy set)
After adding AI features to different models, **XGBoost** had the highest PR-AUC.
- **PR-AUC:** how well the model finds the counties that actually end up with a data center, w/o flagging false positives
- **ROC-AUC:** how good the model is at ranking a random “yes” county above a random “no” county overall (1.0 is perfect)
- **Brier score:** how accurate predicted probabilities are, on average (lower is better)

In [7]:
a1 = summary.loc["+ AI policy"].loc[fam_order]
ref = summary.loc["Reference"]
table_a1 = pd.concat([ref, a1])
table_a1

pr_auc        roc_auc         brier       
                                     mean    std    mean    std   mean    std
model                                                                        
CERF composite suitability ranking  0.187  0.055   0.677  0.049    NaN    NaN
No skill (positive rate)            0.079  0.004   0.500  0.000  0.073  0.004
Logistic regression                 0.336  0.068   0.774  0.060  0.066  0.004
k-nearest neighbors                 0.265  0.074   0.748  0.063  0.072  0.008
Balanced random forest              0.370  0.065   0.801  0.042  0.064  0.009
XGBoost                             0.381  0.062   0.804  0.040  0.062  0.005

## Selected hyperparameters

How often each grid value won the inner search, per family (+ AI policy set). The modal setting is reused
for the fixed hyperparameter analyses in ntbk 07.

In [ ]:
hp = params[params.set == "+ AI policy"]
modal = {}

for fam, cols in [("Logistic regression", ["C"]), ("k-nearest neighbors", ["n_neighbors", "weights"]),
                  ("Balanced random forest", ["max_depth", "min_samples_leaf"]),
                  ("XGBoost", ["max_depth", "learning_rate", "n_estimators"])]:
    choice = hp[hp.model == fam][cols].map(str).agg(", ".join, axis=1)     # map(str) keeps "nan" (= None) as txt
    modal[fam] = dict(zip(cols, choice.mode()[0].split(", ")))
    print(f"{fam}: {choice.value_counts().to_dict()}")

pd.Series(modal).to_json(STAGE4 / "modal_params.json")

Logistic regression: {'0.01': 9, '0.1': 4, '10.0': 1, '1.0': 1}
k-nearest neighbors: {'25.0, distance': 6, '100.0, distance': 4, '10.0, distance': 4, '50.0, distance': 1}
Balanced random forest: {'6.0, 5.0': 4, 'nan, 5.0': 4, '6.0, 1.0': 2, '12.0, 5.0': 2, '12.0, 1.0': 2, 'nan, 1.0': 1}
XGBoost: {'2.0, 0.03, 200.0': 7, '3.0, 0.03, 200.0': 7, '2.0, 0.03, 500.0': 1}


## Table A5 · Thresholds **XGBoost + AI policy**
XGBoost gets further analysis b/c of its high score. This cell tests different cutoff points for calling a county a likely build site. **Averaged** over the three seeds' test folds.
- low cutoff **(0.05)** flags way more counties and catches more real ones, but w/ many false alarms
- high cutoff **(0.50)** flags fewer counties but misses more 

In [ ]:
best = oof[(oof.model == "XGBoost") & (oof.set == "+ AI policy")]
rows = []

for t in [0.05, 0.10, 0.20, 0.30, 0.50]:
    per_seed = []
    for s, g in best.groupby("seed"):
        flag = g.prob >= t
        tp = (flag & (g.y == 1)).sum()
        per_seed.append({"precision": tp / max(flag.sum(), 1), "recall": tp / g.y.sum(), "flagged": flag.sum()})
    rows.append({"threshold": t, **pd.DataFrame(per_seed).mean().round(3).to_dict()})

thresholds = pd.DataFrame(rows)
thresholds.to_csv(STAGE4 / "thresholds.csv", index=False)
thresholds

,threshold,precision,recall,flagged
0,0.05,0.165,0.761,1128.000
1,0.10,0.236,0.626,653.667
2,0.20,0.344,0.433,308.333
3,0.30,0.431,0.298,170.333
4,0.50,0.510,0.165,78.000


## Calibration check **XGBoost + AI policy**

*Can the model’s probabilities can be trusted at face value?* Following table shows **mean predicted probability** against the observed rate, in ten probability bins.
- for ex: seeing if the bin the model calls “about 3% chance” actually had a site about 3% of the time.

In [10]:
bins = pd.qcut(best.prob.rank(method="first"), 10, labels=False)
best.groupby(bins).agg(mean_predicted=("prob", "mean"), observed=("y", "mean"), counties=("y", "size")).round(3)

,mean_predicted,observed,counties
prob,,,
0,0.007,0.014,930
1,0.021,0.035,930
2,0.025,0.028,929
3,0.026,0.023,930
4,0.029,0.025,930
5,0.036,0.045,929
6,0.053,0.039,930
7,0.085,0.098,929
8,0.141,0.141,930
